# Deliberative Architecture
## Customer Support Agent – Planning-Based Design

---

### What you will learn

- What a **Deliberative Agent** is
- How **planning before acting** improves intelligence
- How a **dispatcher pattern** maps step names to real functions
- How **shared state** connects steps together
- Why deliberative systems are powerful but slow

Use case (unchanged):
> **E-commerce Customer Support Agent**

## 1. Deliberative Architecture Overview

A deliberative agent follows the loop:

```
OBSERVE → PLAN → EXECUTE
```

Unlike reactive agents, deliberative agents **think before acting**.

They explicitly construct a **plan** to reach a goal.

```
USER QUERY
    ↓
  OBSERVE        ← sense the environment
    ↓
   PLAN          ← decide the sequence of steps
    ↓
  EXECUTE        ← dispatch each step via registry
    ↓    ↓    ↓    ↓
  ask  fetch check respond   ← individual step functions
    ↓
  RESPONSE
```

**Contrast with Notebook_1 (Reactive):** In reactive architecture, sense leads directly to act with no intermediate reasoning. Here, a dedicated planning stage sits in between.

## 2. Industry Intuition

Deliberative architectures are useful when:

- Tasks require **multiple steps**
- Actions must follow a **logical sequence**
- Mistakes are expensive

### Real-world examples
- Workflow engines (Apache Airflow — each task name maps to a callable, just like our `step_registry`)
- IT service management systems
- Order resolution pipelines

## 3. Problem Statement

User asks:

> "Where is my order?"

The agent must:
1. Ask for order ID
2. Fetch order details from the orders database
3. Check shipping status
4. Respond with status

These steps form a **plan**.

In [5]:
import pandas as pd

# Load the orders knowledge base once — shared across all step functions
orders_df = pd.read_csv("order.csv")
print(f"Loaded {len(orders_df)} orders")
print(orders_df.head(3))

Loaded 30 orders
   order_id   user_name                 product_name      status        date
0      5001    John Doe               Wireless Mouse   Delivered  2024-08-15
1      5002  Jane Smith              Gaming Keyboard     Shipped  2024-08-20
2      5003    John Doe  Noise Cancelling Headphones  Processing  2024-08-21


## The Dispatcher Pattern

The dispatcher is the heart of deliberative execution.

```python
step_registry = {
    "ask_order_id":   ask_order_id,
    "fetch_order":    fetch_order,
    "check_shipping": check_shipping,
    "respond_to_user": respond_to_user,
}
```

`plan()` returns a list of strings like `["ask_order_id", "fetch_order", ...]`.  
`execute()` loops over that list and looks up each string as a key in the registry.

This **decouples planning from execution**:
- `plan()` does not know *how* steps work — only their names
- `execute()` does not know *what* step to run next — only how to run any step
- Step functions do not know about each other at all

This is **separation of concerns** made concrete.

## The Shared State Dictionary

Each step function receives and mutates a single `state` dict — the agent's **short-term working memory** for one conversation turn.

```
state = {
    "user_message":    "...",   # set before the loop starts
    "order_id":        None,    # written by ask_order_id
    "order_record":    None,    # written by fetch_order
    "shipping_status": None,    # written by check_shipping
    "shipping_message": None,   # written by check_shipping
    "response":        None     # written by respond_to_user (final output)
}
```

Each step reads keys written by earlier steps and writes keys for later steps.  
There are no hidden dependencies — every data flow is explicit.

In [10]:
import re

def get_order_id(state: dict) -> None:
    """Step 1: Extract the order ID from the user's message."""
    match = re.search(r'\b(50\d{2})\b', state.get("user_message", ""))
    if match:
        state["order_id"] = int(match.group(1))
        print(f"  [get_order_id]    extracted order_id = {state['order_id']}")
    else:
        state["order_id"] = None
        print("  [get_order_id]    no order ID found in message")


def fetch_order(state: dict) -> None:
    """Step 2: Retrieve the order record from the CSV knowledge base."""
    oid = state.get("order_id")
    if oid is None:
        state["order_record"] = None
        print("  [fetch_order]     ERROR: no order_id in state")
        return
    # order_id in CSV is int64 — oid must also be int to avoid silent type mismatch
    result = orders_df[orders_df["order_id"] == int(oid)]
    state["order_record"] = result.iloc[0].to_dict() if not result.empty else None
    print(f"  [fetch_order]     {state['order_record']}")


def check_shipping(state: dict) -> None:
    """Step 3: Extract and interpret the shipping status."""
    record = state.get("order_record")
    if not record:
        state["shipping_status"] = "UNKNOWN"
        print("  [check_shipping]  No order record available")
        return
    status_messages = {
        "Delivered":  "Your order has been delivered.",
        "Shipped":    "Your order is on the way.",
        "Processing": "Your order is being prepared.",
        "Cancelled":  "Your order was cancelled.",
    }
    state["shipping_status"] = record["status"]
    state["shipping_message"] = status_messages.get(record["status"], "Status unknown.")
    print(f"  [check_shipping]  {record['status']} → '{state['shipping_message']}'")


def respond_to_user(state: dict) -> None:
    """Step 4: Compose the final response from accumulated state."""
    r = state.get("order_record")
    msg = state.get("shipping_message", "We could not retrieve your order.")
    state["response"] = (
        f"Hi {r['user_name']}, your {r['product_name']} "
        f"(Order #{r['order_id']}, {r['date']}): {msg}"
        if r else "We could not find your order. Please check the order ID."
    )
    print(f"  [respond_to_user] {state['response']}")


def generic_response(state: dict) -> None:
    """Fallback step for unrecognized queries."""
    state["response"] = "I can help with orders, refunds, or delivery questions."
    print(f"  [generic_response] {state['response']}")

In [11]:
def plan(user_message: str) -> list:
    """
    WHAT to do: returns a list of step names based on intent.
    Does not know how steps work — only their names.
    """
    if "order" in user_message.lower():
        return ["get_order_id", "fetch_order", "check_shipping", "respond_to_user"]
    return ["generic_response"]


# The registry bridges the plan (data) and the executor (logic)
step_registry = {
    "get_order_id":   get_order_id,
    "fetch_order":    fetch_order,
    "check_shipping": check_shipping,
    "respond_to_user": respond_to_user,
    "generic_response": generic_response,
}


def execute(plan_steps: list, state: dict) -> None:
    """
    HOW to do it: dispatches each step name to its function via the registry.
    Does not know what the plan is — only how to dispatch any step.
    """
    for step in plan_steps:
        print(f"\n--- Step: {step} ---")
        if step in step_registry:
            step_registry[step](state)
        else:
            print(f"  WARNING: unknown step '{step}' — skipping")

In [12]:
# --- Happy path: order ID embedded in the query ---
user_query = "Where is my order 5003?"

state = {
    "user_message": user_query,
    "order_id": None,
    "order_record": None,
    "shipping_status": None,
    "shipping_message": None,
    "response": None,
}

steps = plan(user_query)
print(f"USER:  {user_query}")
print(f"PLAN:  {steps}")
print("\nEXECUTION:")
execute(steps, state)
print(f"\nFINAL RESPONSE: {state['response']}")

USER:  Where is my order 5003?
PLAN:  ['get_order_id', 'fetch_order', 'check_shipping', 'respond_to_user']

EXECUTION:

--- Step: get_order_id ---
  [get_order_id]    extracted order_id = 5003

--- Step: fetch_order ---
  [fetch_order]     {'order_id': 5003, 'user_name': 'John Doe', 'product_name': 'Noise Cancelling Headphones', 'status': 'Processing', 'date': '2024-08-21'}

--- Step: check_shipping ---
  [check_shipping]  Processing → 'Your order is being prepared.'

--- Step: respond_to_user ---
  [respond_to_user] Hi John Doe, your Noise Cancelling Headphones (Order #5003, 2024-08-21): Your order is being prepared.

FINAL RESPONSE: Hi John Doe, your Noise Cancelling Headphones (Order #5003, 2024-08-21): Your order is being prepared.


In [13]:
# --- Fallback path: non-order query ---
user_query_2 = "What is your return policy?"

state_2 = {"user_message": user_query_2, "response": None}
steps_2 = plan(user_query_2)

print(f"USER: {user_query_2}")
print(f"PLAN: {steps_2}")
print("\nEXECUTION:")
execute(steps_2, state_2)
print(f"\nFINAL RESPONSE: {state_2['response']}")

USER: What is your return policy?
PLAN: ['generic_response']

EXECUTION:

--- Step: generic_response ---
  [generic_response] I can help with orders, refunds, or delivery questions.

FINAL RESPONSE: I can help with orders, refunds, or delivery questions.


In [14]:
# --- Multiple queries: each message carries its own order ID ---
# get_order_id extracts the ID from the message — no manual pre-setting needed
queries = [
    "Where is my order 5002?",
    "Can you check order 5005 for me?",
    "I need an update on order 5007",
    "Track my order 5020 please",
]

for query in queries:
    print(f"\n{'='*55}")
    print(f"USER: {query}")
    state = {
        "user_message": query,
        "order_id": None,
        "order_record": None,
        "shipping_status": None,
        "shipping_message": None,
        "response": None,
    }
    execute(plan(query), state)
    print(f"RESPONSE: {state['response']}")


USER: Where is my order 5002?

--- Step: get_order_id ---
  [get_order_id]    extracted order_id = 5002

--- Step: fetch_order ---
  [fetch_order]     {'order_id': 5002, 'user_name': 'Jane Smith', 'product_name': 'Gaming Keyboard', 'status': 'Shipped', 'date': '2024-08-20'}

--- Step: check_shipping ---
  [check_shipping]  Shipped → 'Your order is on the way.'

--- Step: respond_to_user ---
  [respond_to_user] Hi Jane Smith, your Gaming Keyboard (Order #5002, 2024-08-20): Your order is on the way.
RESPONSE: Hi Jane Smith, your Gaming Keyboard (Order #5002, 2024-08-20): Your order is on the way.

USER: Can you check order 5005 for me?

--- Step: get_order_id ---
  [get_order_id]    extracted order_id = 5005

--- Step: fetch_order ---
  [fetch_order]     {'order_id': 5005, 'user_name': 'Jane Smith', 'product_name': '4K Monitor', 'status': 'Cancelled', 'date': '2024-08-05'}

--- Step: check_shipping ---
  [check_shipping]  Cancelled → 'Your order was cancelled.'

--- Step: respond_to_use

In [15]:
# --- Inspect the agent's working memory after the last execution ---
import json
print("Final state after execution:")
print(json.dumps(state, indent=2, default=str))

Final state after execution:
{
  "user_message": "Track my order 5020 please",
  "order_id": 5020,
  "order_record": {
    "order_id": 5020,
    "user_name": "Ella Perez",
    "product_name": "Action Camera",
    "status": "Delivered",
    "date": "2024-08-11"
  },
  "shipping_status": "Delivered",
  "shipping_message": "Your order has been delivered.",
  "response": "Hi Ella Perez, your Action Camera (Order #5020, 2024-08-11): Your order has been delivered."
}


## 4. Key Observations

- The plan is created **before execution**
- Execution strictly follows the plan via the **dispatcher registry**
- **State** is the agent's working memory — every step reads what earlier steps wrote
- The **plan is variable**: the multi-order loop skips `ask_order_id` because the ID is already known — the planner adapts to context
- Adding a new capability requires only: one new function + one registry entry — no changes to `plan()` or `execute()`

Try modifying the plan order or adding a new step function and re-run.

## 5. Strengths of Deliberative Architecture

✅ Clear reasoning path  
✅ Predictable behavior  
✅ Suitable for structured workflows

This makes deliberative agents common in **enterprise systems**.

## 6. Weaknesses of Deliberative Architecture

❌ Slow (planning cost)  
❌ Brittle when assumptions fail  
❌ Poor fit for dynamic environments

If a step fails, the entire plan may become invalid.

## 7. Comparison with Reactive Architecture

| Aspect | Reactive | Deliberative |
|------|---------|--------------|
| Speed | Very fast | Slower |
| Planning | None | Explicit |
| Memory | None | Plan memory |
| State sharing | None | Shared context dict |
| Adaptability | Low | Low–Medium |